In [1]:
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

import torch, transformers, trl, peft, pymatgen.core
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| trl", trl.__version__, "| peft", peft.__version__)

print("MPS available:", torch.backends.mps.is_available())
x = torch.ones(3, device="mps")
print("Test computation on GPU:", (x * 2).tolist())

torch 2.14.0 | transformers 5.17.0 | trl 1.13.0 | peft 0.21.0
MPS available: True
Test computation on GPU: [2.0, 2.0, 2.0]


In [2]:
# Pull the final chemical formula out of a model response.
# Handles markdown formatting (**Formula:**) and subscript digits (Li₂MnO₃).
import re

SUBSCRIPTS = str.maketrans("₀₁₂₃₄₅₆₇₈₉−", "0123456789-")

def normalize(text):
    return text.translate(SUBSCRIPTS)

def extract_formula(text):
    text = normalize(text)
    matches = re.findall(r"[Ff]ormula:[\s*`]*([A-Z][A-Za-z0-9()]*)", text)
    return matches[-1] if matches else None

In [3]:
# The reward function. Scores a response as -1 (unusable),
# 0 (valid formula, not charge-balanced), or 1 (charge-balanced, correct elements).
from pymatgen.core import Composition

def score_response(text, required_elements):
    """Return -1 if unusable, 0 if valid but not charge-balanced, 1 if balanced."""
    formula = extract_formula(text)
    if formula is None:
        return -1
    try:
        comp = Composition(formula)
    except Exception:
        return -1
    if {el.symbol for el in comp.elements} != set(required_elements):
        return -1
    return 1 if comp.oxi_state_guesses() else 0

In [5]:
# Check whether the oxidation states the model stated in its
# reasoning actually sum to zero for the formula it gave. Used for analysis, not reward.
def stated_states_match(text):
    text = normalize(text)
    formula = extract_formula(text)
    states = dict(re.findall(r"\b([A-Z][a-z]?):\s*([+-]?\d+)", text))
    if formula is None:
        return False
    try:
        comp = Composition(formula)
    except Exception:
        return False
    symbols = {el.symbol for el in comp.elements}
    if not symbols <= set(states):
        return False
    total = sum(comp[s] * int(states[s]) for s in symbols)
    return abs(total) < 1e-6

In [ ]:
# Test the scorer and faithfulness check on cases with known answers.
# All eight lines should print PASS before moving on to Phase 2.
GOOD = "Li: +1, Mn: +4, O: -2\nCheck: 2(+1) + (+4) + 3(-2) = 0\nFormula: Li2MnO3"
REQ = ["Li", "Mn", "O"]

tests = [
    ("balanced",             score_response(GOOD, REQ),                          1),
    ("markdown bold",        score_response("**Formula:** Li2MnO3", REQ),        1),
    ("unicode subscripts",   score_response("Formula: Li₂MnO₃", REQ),            1),
    ("unbalanced, Mn +9",    score_response("Formula: LiMnO5", REQ),             0),
    ("wrong elements",       score_response("Formula: H2O", REQ),               -1),
    ("no formula line",      score_response("Li2MnO3 is nice", REQ),            -1),
    ("faithful reasoning",   stated_states_match(GOOD),                          True),
    ("unfaithful reasoning", stated_states_match(GOOD.replace("Mn: +4", "Mn: +2")), False),
]
for name, got, expected in tests:
    print("PASS" if got == expected else "FAIL", "|", name, "| expected", expected, "got", got)

PASS | balanced | expected 1 got 1
PASS | markdown bold | expected 1 got 1
PASS | unicode subscripts | expected 1 got 1
PASS | unbalanced, Mn +9 | expected 0 got 0
PASS | wrong elements | expected -1 got -1
PASS | no formula line | expected -1 got -1
PASS | faithful reasoning | expected True got True
PASS | unfaithful reasoning | expected False got False


In [2]:
# Define the element pools and prompt template, then form
# every combination of one cation, one transition metal, and one anion (220 total).
import itertools, json, os, random
from pymatgen.core import Element

CATIONS = ["Li", "Na", "K", "Mg", "Ca"]
METALS  = ["Ti", "V", "Cr", "Mn", "Fe", "Co", "Ni", "Cu", "Zn", "Nb", "Mo"]
ANIONS  = ["O", "S", "F", "N"]

TEMPLATE = (
    "Propose a plausible inorganic compound containing {a}, {b}, and {c}.\n"
    "First state the oxidation state you will use for each element.\n"
    "Then give the compound on a final line in the form 'Formula: <formula>'."
)

def name(symbol):
    return Element(symbol).long_name.lower()

combos = list(itertools.product(CATIONS, METALS, ANIONS))
print(len(combos), "element combinations")

220 element combinations


In [ ]:
# Shuffle with a fixed seed, hold out one fifth for testing,
# and save both splits to the data/ folder so every later phase uses the same prompts.
random.seed(0)
random.shuffle(combos)
n_test = len(combos) // 5
test_combos, train_combos = combos[:n_test], combos[n_test:]

def to_record(combo):
    a, b, c = combo
    return {"elements": list(combo), "prompt": TEMPLATE.format(a=name(a), b=name(b), c=name(c))}

os.makedirs("data", exist_ok=True)
for split, items in [("train", train_combos), ("test", test_combos)]:
    with open(f"data/{split}_prompts.json", "w") as f:
        json.dump([to_record(c) for c in items], f, indent=2)

print(len(train_combos), "train |", len(test_combos), "test")
print(to_record(test_combos[0])["prompt"])

176 train | 44 test
Propose a plausible inorganic compound containing calcium, copper, and fluorine.
First state the oxidation state you will use for each element.
Then give the compound on a final line in the form 'Formula: <formula>'.


In [ ]:
# Load the 1.5B model and tokenizer from the local cache onto the Apple GPU.
# The files were downloaded separately with `hf download`; local_files_only avoids any network access.
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

DEVICE = "mps"
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL_NAME, local_files_only=True)
tok.padding_side = "left"
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float32,
                                             local_files_only=True).to(DEVICE)
model.eval()
print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.0f}M parameters on {model.device}")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

1544M parameters on mps:0


In [4]:
# Generate several responses per prompt, in batches.
# Each prompt is preceded by one worked example so the model sees the exact answer format.
EXAMPLE_Q = TEMPLATE.format(a="strontium", b="aluminum", c="oxygen")
EXAMPLE_A = "Sr: +2, Al: +3, O: -2\nCheck: (+2) + 2(+3) + 4(-2) = 0\nFormula: SrAl2O4"

def build_input(prompt):
    messages = [{"role": "user", "content": EXAMPLE_Q},
                {"role": "assistant", "content": EXAMPLE_A},
                {"role": "user", "content": prompt}]
    return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def generate(prompts, n_samples=8, batch_prompts=4, max_new_tokens=150):
    results = []
    for i in range(0, len(prompts), batch_prompts):
        batch = [build_input(p) for p in prompts[i:i + batch_prompts]]
        inputs = tok(batch, return_tensors="pt", padding=True).to(DEVICE)
        with torch.no_grad():
            out = model.generate(**inputs, do_sample=True, num_return_sequences=n_samples,
                                 max_new_tokens=max_new_tokens, pad_token_id=tok.pad_token_id)
        texts = tok.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        results += [texts[j:j + n_samples] for j in range(0, len(texts), n_samples)]
        print(f"{min(i + batch_prompts, len(prompts))}/{len(prompts)} prompts done")
    return results

print(model.generation_config)

GenerationConfig {
  "bos_token_id": 151643,
  "do_sample": true,
  "eos_token_id": [
    151645,
    151643
  ],
  "pad_token_id": 151643,
  "repetition_penalty": 1.1,
  "temperature": 0.7,
  "top_k": 20,
  "top_p": 0.8
}



In [ ]:
# Check the format on three prompts, four responses each.
# A broader sample than before, to judge whether the fix works before the full run.
set_seed(0)
test = json.load(open("data/test_prompts.json"))
groups = generate([t["prompt"] for t in test[:3]], n_samples=4)
for t, group in zip(test[:3], groups):
    for r in group:
        print(r, "\n-> score:", score_response(r, t["elements"]), "\n" + "-" * 40)

3/3 prompts done
Ca: +2, Cu: +1, F: -1
Check: (+2) + (+1) + (-1) x 5 = 0
Formula: CaCuF5 
-> score: 1 
----------------------------------------
Ca: +2, Cu: +2, F: -1
Check: (+2) + 2(+2) + (-1) = 0
Formula: CaCuF2 
-> score: 0 
----------------------------------------
Ca: +2, Cu: +1, F: -1
Check: (+2) + (+1) + (-1) = 0
Formula: CaCuF2 
-> score: 0 
----------------------------------------
Ca: +2, Cu: +1, F: -1
Check: (+2) + (+1) + 4(-1) = 0
Formula: CaCuF2 
-> score: 0 
----------------------------------------
Na: +1, Mn: +7, F: -1
Check: (+1) + 7(+7) + (-1)(4) = 0
Formula: NaMnF4 
-> score: 1 
----------------------------------------
Na: +1, Mn: +7, F: -1
Check: (+1) + 7(+7) + (-1)(4) = 0
Formula: NaMF4 
-> score: -1 
----------------------------------------
Na: +1, Mn: +7, F: -1
Check: (+1) + 7(+7) + 5(-1) = 0
Formula: NaMnF5 
-> score: 1 
----------------------------------------
Na: +1, Mn: +7, F: -1
Check: (+1) + 7(+7) + (-1)(4) = 0
Formula: NaMnF4 
-> score: 1 
--------------------

In [ ]:
#Qwen 0.5B systematically omitted the transition metal, collapsing to familiar binary compounds. This could be due to 
#lack of TMO_x in training data, so instead using 1.5B

#interestingly, the 1.5B model seems to output a correct compound but fails toprovide the right oxidation reasoning. so the
#experiment here becomes will the dpo actually make the model's oxidation reasoning better (arguably more important) or just
#raise formula accuracy merely.

In [ ]:
# Generate 8 responses for each of the 44 held-out prompts (352 total)
# and save them, so the metrics can be recomputed or inspected later without regenerating.
set_seed(0)
base_responses = generate([t["prompt"] for t in test])
with open("data/baseline_responses.json", "w") as f:
    json.dump([{"elements": t["elements"], "responses": r} for t, r in zip(test, base_responses)],
              f, indent=2)

4/44 prompts done
8/44 prompts done
12/44 prompts done
16/44 prompts done
20/44 prompts done
24/44 prompts done
28/44 prompts done
32/44 prompts done
36/44 prompts done
40/44 prompts done
44/44 prompts done


In [13]:
# Compute validity, balance, diversity, and faithfulness from the saved responses.
# These are the baseline numbers the trained model will be compared against in Phase 6.
def metrics(records):
    scores, faithful, diversity = [], [], []
    for rec in records:
        formulas = [extract_formula(r) for r in rec["responses"]]
        diversity.append(len({f for f in formulas if f}) / len(formulas))
        for r in rec["responses"]:
            s = score_response(r, rec["elements"])
            scores.append(s)
            if s == 1:
                faithful.append(stated_states_match(r))
    n = len(scores)
    return {"validity": sum(s >= 0 for s in scores) / n,
            "balance": sum(s == 1 for s in scores) / n,
            "diversity": sum(diversity) / len(diversity),
            "faithfulness": sum(faithful) / max(len(faithful), 1),
            "n_responses": n}

with open("data/baseline_responses.json") as f:
    baseline = metrics(json.load(f))
with open("data/baseline_metrics.json", "w") as f:
    json.dump(baseline, f, indent=2)
print(baseline)

{'validity': 0.8465909090909091, 'balance': 0.5596590909090909, 'diversity': 0.3465909090909091, 'faithfulness': 0.3147208121827411, 'n_responses': 352}


Results (8 responses each for 44 prompts):

Validity is 84.7%: the answer is a usable formula with exactly the requested elements.

Balance is 56%: The answer was scored correct.

Diversity is 35%: about 2.8 of distinct answers per 8 attempts at the same question.

Faithfulness is 31.5%: of the correct answers, only that many had valid reasoning.

In [ ]:
# Print five balanced responses that the faithfulness check marked unfaithful.
# Confirms the low faithfulness rate reflects real reasoning errors, not a parsing problem.
with open("data/baseline_responses.json") as f:
    records = json.load(f)
shown = 0
for rec in records:
    for r in rec["responses"]:
        if shown < 5 and score_response(r, rec["elements"]) == 1 and not stated_states_match(r):
            print(r, "\n" + "-" * 40)
            shown += 1

Ca: +2, Cu: +1, F: -1
Check: (+2) + (+1) + (-1) × 5 = 0
Formula: CaCuF5 
----------------------------------------
Ca: +2, Cu: +1, F: -1
Check: (+2) + (+1) + (-1)(4) = 0
Formula: CaCuF4 
----------------------------------------
Na: +1, Mn: +7, F: -1
Check: (+1) + 7(+7) + (-1)(4) = 0
Formula: NaMnF4 
----------------------------------------
Na: +1, Mn: +7, F: -1
Check: (+1) + 7(+7) + (-1)(4) = 0
Formula: NaMnF4 
----------------------------------------
Na: +1, Mn: +7, F: -1
Check: (+1) + 7(+7) + 4(-1) = 0
Formula: Na2MnF4 
----------------------------------------


In [ ]:
# Display the baseline metrics as a readable table, loaded from the saved file
# so this works even after a kernel restart.
import pandas as pd

with open("data/baseline_metrics.json") as f:
    baseline = json.load(f)
labels = {"validity": "Validity: usable formula with the requested elements",
          "balance": "Balance: answer scored correct",
          "diversity": "Diversity: distinct answers per question",
          "faithfulness": "Faithfulness: correct answers supported by stated reasoning"}
table = pd.DataFrame({"Metric": list(labels.values()),
                      "Baseline": [f"{baseline[k]:.1%}" for k in labels]}).set_index("Metric")
display(table)

,Baseline
Metric,
Validity: usable formula with the requested elements,84.7%
Balance: answer scored correct,56.0%
Diversity: distinct answers per question,34.7%
Faithfulness: correct answers supported by stated reasoning,31.5%


In [ ]:
# Generate 8 responses for each of the 176 training prompts (1,408 total), with the same
# settings as the baseline. Saves after every 16 prompts, so an interruption loses little.
with open("data/train_prompts.json") as f:
    train = json.load(f)
path = "data/train_responses.json"
done = json.load(open(path)) if os.path.exists(path) else []

set_seed(0)
for start in range(len(done), len(train), 16):
    chunk = train[start:start + 16]
    groups = generate([t["prompt"] for t in chunk])
    done += [{"elements": t["elements"], "prompt": t["prompt"], "responses": g}
             for t, g in zip(chunk, groups)]
    with open(path, "w") as f:
        json.dump(done, f, indent=2)
    print(f"Saved {len(done)} of {len(train)} prompts")

In [ ]:
# Score every training response and form preference pairs: for each prompt, a correct
# response (score 1) is preferred over an incorrect one (0 or -1), at most 4 pairs per prompt.
with open("data/train_responses.json") as f:
    train_records = json.load(f)

rng = random.Random(0)
pairs, outcomes, rejected_scores = [], [], []
for rec in train_records:
    scored = {r: score_response(r, rec["elements"]) for r in rec["responses"]}
    winners = [r for r, s in scored.items() if s == 1]
    losers = [r for r, s in scored.items() if s < 1]
    if not winners or not losers:
        outcomes.append("All incorrect: no pairs" if not winners else "All correct: no pairs")
        continue
    outcomes.append("Mixed: pairs formed")
    prompt = [{"role": "user", "content": EXAMPLE_Q}, {"role": "assistant", "content": EXAMPLE_A},
              {"role": "user", "content": rec["prompt"]}]
    combos = list(itertools.product(winners, losers))
    rng.shuffle(combos)
    for w, l in combos[:4]:
        pairs.append({"prompt": prompt, "chosen": [{"role": "assistant", "content": w}],
                      "rejected": [{"role": "assistant", "content": l}]})
        rejected_scores.append(scored[l])

with open("data/dpo_pairs.json", "w") as f:
    json.dump(pairs, f, indent=2)
print(len(pairs), "pairs saved")

274 pairs saved


In [ ]:
# Summarize the pairs: how the training prompts split by outcome, and what kind of
# response was rejected in each pair.
by_prompt = pd.Series(outcomes).value_counts().rename_axis("Training prompts").to_frame("Count")
by_prompt["Share"] = (by_prompt["Count"] / by_prompt["Count"].sum()).map("{:.0%}".format)
display(by_prompt)

names = {0: "Valid formula, not balanced (score 0)", -1: "Unusable or wrong elements (score -1)"}
by_pair = pd.Series(rejected_scores).map(names).value_counts().rename_axis("Rejected response").to_frame("Pairs")
by_pair.loc["Total"] = by_pair["Pairs"].sum()
display(by_pair)

,Count,Share
Training prompts,,
Mixed: pairs formed,85,48%
All correct: no pairs,66,38%
All incorrect: no pairs,25,14%


,Pairs
Rejected response,
"Valid formula, not balanced (score 0)",157
Unusable or wrong elements (score -1),117
Total,274


In [ ]:
# Print three random pairs to confirm by eye that each chosen response really is
# correct and each rejected response really is worse.
for p in random.Random(1).sample(pairs, 3):
    print("QUESTION:", p["prompt"][-1]["content"].splitlines()[0])
    print("\nCHOSEN:\n" + p["chosen"][0]["content"])
    print("\nREJECTED:\n" + p["rejected"][0]["content"])
    print("=" * 60)

QUESTION: Propose a plausible inorganic compound containing sodium, titanium, and sulfur.

CHOSEN:
Na: +1, Ti: +4, S: -2
Check: (+1) + 4(+4) + (-2) = 0
Formula: Na2TiS2

REJECTED:
Na: +1, Ti: +4, S: -2
Check: (+1) + 4(+4) + (-2) = 0
Formula: Na4Ti6S2
QUESTION: Propose a plausible inorganic compound containing lithium, vanadium, and nitrogen.

CHOSEN:
Li: +1, V: +5, N: -3
Check: (+1) + 5(+5) + (-3) * 4 = 0
Formula: LiV2N4

REJECTED:
Li: +1, V: +5, N: -3
Check: (+1) + 5(+5) + 3(-3) = 0
Formula: LiV3N
QUESTION: Propose a plausible inorganic compound containing magnesium, zinc, and nitrogen.

CHOSEN:
Mg: +2, Zn: +2, N: -3
Check: (+2) + 2(+2) + (-3) = 1
Formula: Mg2ZnN3

REJECTED:
Mg: +2, Zn: +2, N: -3
Check: (+2) + 2(+2) + (-3) = 1
Formula: MgZnN


In [6]:
# Fresh-kernel setup for training: enable the Apple GPU fallback, then load the
# 274 preference pairs into the dataset format TRL expects.
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
import json, torch
import pandas as pd
from datasets import Dataset

with open("data/dpo_pairs.json") as f:
    train_dataset = Dataset.from_list(json.load(f))
print(train_dataset)

Dataset({
    features: ['prompt', 'chosen', 'rejected'],
    num_rows: 274
})


In [ ]:
# Load the 1.5B model and tokenizer from the local cache. TRL will attach LoRA to this
# model and use the same model with LoRA switched off as the frozen reference.
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL_NAME, local_files_only=True)
tok.padding_side = "left"
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float32,
                                             local_files_only=True)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [ ]:
# Training settings. LoRA trains small added matrices in the attention layers;
# DPOConfig sets beta, the learning rate, batch sizes, and Apple-GPU-safe options.
from peft import LoraConfig
from trl import DPOConfig, DPOTrainer

lora_config = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.0, task_type="CAUSAL_LM",
                         target_modules=["q_proj", "k_proj", "v_proj", "o_proj"])
args = DPOConfig(
    output_dir="checkpoints/dpo_beta0.1",
    beta=0.1,
    learning_rate=2e-5,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    warmup_steps=5,
    max_length=512,
    logging_steps=5,
    save_strategy="epoch",
    bf16=False,
    optim="adamw_torch",
    dataloader_pin_memory=False,
    seed=0,
)

In [ ]:
# Create the trainer and run DPO. The trained LoRA
# adapter is saved to adapters/.
trainer = DPOTrainer(model=model, args=args, train_dataset=train_dataset,
                     processing_class=tok, peft_config=lora_config)
trainer.model.print_trainable_parameters()
trainer.train()
trainer.save_model("adapters/dpo_beta0.1")

Tokenizing train dataset:   0%|          | 0/274 [00:00<?, ? examples/s]

Dropping fully truncated examples from train dataset:   0%|          | 0/274 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815


Step,Training Loss
5,0.692014
10,0.673178
15,0.661076
20,0.638926
25,0.604674
30,0.551865
35,0.555276
40,0.565467
45,0.519802
50,0.498485


In [ ]:
# Show the logged training metrics as a table, one row per logging step.
cols = ["step", "loss", "rewards/chosen", "rewards/rejected", "rewards/margins", "rewards/accuracies"]
log = pd.DataFrame([r for r in trainer.state.log_history if "rewards/margins" in r])
display(log[cols].set_index("step").round(3))

,loss,rewards/chosen,rewards/rejected,rewards/margins,rewards/accuracies
step,,,,,
5,0.692,-0.002,-0.004,0.002,0.588
10,0.673,0.008,-0.033,0.041,0.825
15,0.661,0.012,-0.056,0.068,0.800
20,0.639,0.018,-0.128,0.146,0.864
25,0.605,0.023,-0.176,0.199,0.875
30,0.552,0.021,-0.313,0.334,0.900
35,0.555,-0.015,-0.345,0.330,0.838
40,0.565,0.016,-0.367,0.384,0.864
45,0.520,-0.014,-0.454,0.440,0.900


In [3]:
# Load a fresh copy of the original 1.5B model and attach the trained DPO adapter.
# merge_and_unload folds the LoRA matrices into the weights so generation runs at full speed.
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from peft import PeftModel

DEVICE = "mps"
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL_NAME, local_files_only=True)
tok.padding_side = "left"
base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float32, local_files_only=True)
model = PeftModel.from_pretrained(base, "adapters/dpo_beta0.1").merge_and_unload().to(DEVICE)
model.eval()

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1536,), eps=1e-06)
    (rotar

In [11]:
# Evaluate the trained model: 8 responses for each of the 44 held-out prompts, with exactly
# the baseline's settings and seed, saved for the metrics and for the credit analysis.
with open("data/test_prompts.json") as f:
    test = json.load(f)
set_seed(0)
dpo_responses = generate([t["prompt"] for t in test])
with open("data/dpo_responses.json", "w") as f:
    json.dump([{"elements": t["elements"], "responses": r} for t, r in zip(test, dpo_responses)],
              f, indent=2)


4/44 prompts done
8/44 prompts done
12/44 prompts done
16/44 prompts done
20/44 prompts done
24/44 prompts done
28/44 prompts done
32/44 prompts done
36/44 prompts done
40/44 prompts done
44/44 prompts done


In [14]:
# Compare the baseline and trained models on every metric, including balance among valid
# answers, which separates better chemistry from better instruction-following.
import pandas as pd

with open("data/baseline_responses.json") as f:
    before = metrics(json.load(f))
with open("data/dpo_responses.json") as f:
    after = metrics(json.load(f))
for m in (before, after):
    m["balance_among_valid"] = m["balance"] / m["validity"]
with open("data/dpo_metrics.json", "w") as f:
    json.dump(after, f, indent=2)

labels = {"validity": "Validity: usable formula with the requested elements",
          "balance": "Balance: answer scored correct",
          "balance_among_valid": "Balance among valid answers",
          "diversity": "Diversity: distinct answers per question",
          "faithfulness": "Faithfulness: correct answers supported by stated reasoning"}
table = pd.DataFrame({"Metric": list(labels.values()),
                      "Baseline": [f"{before[k]:.1%}" for k in labels],
                      "After DPO": [f"{after[k]:.1%}" for k in labels],
                      "Change": [f"{(after[k] - before[k]) * 100:+.1f} pts" for k in labels]}
                     ).set_index("Metric")
display(table)

,Baseline,After DPO,Change
Metric,,,
Validity: usable formula with the requested elements,84.7%,92.3%,+7.7 pts
Balance: answer scored correct,56.0%,71.3%,+15.3 pts
Balance among valid answers,66.1%,77.2%,+11.1 pts
Diversity: distinct answers per question,34.7%,25.6%,-9.1 pts
Faithfulness: correct answers supported by stated reasoning,31.5%,22.7%,-8.8 pts


In [15]:
# confirming reasoning drop is real instead of a formatting issue
with open("data/dpo_responses.json") as f:
    records = json.load(f)
shown = 0
for rec in records:
    for r in rec["responses"]:
        if shown < 5 and score_response(r, rec["elements"]) == 1 and not stated_states_match(r):
            print(r, "\n" + "-" * 40)
            shown += 1

Ca: +2, Cu: +1, F: -1
Check: (+2) + (+1) + (-1)(4) = 0
Formula: CaCuF4 
----------------------------------------
Na: +1, Mn: +7, F: -1
Check: (+1) + 7(+7) + 4(-1) = 0
Formula: NaMnF4 
----------------------------------------
Na: +1, Mn: +7, F: -1
Check: (+1) + 7(+7) + 4(-1) = 0
Formula: NaMnF4 
----------------------------------------
Na: +1, Mn: +2, F: -1
Check: (+1) + 2(+2) + 4(-1) = 0
Formula: NaMnF4 
----------------------------------------
Na: +1, Mn: +7, F: -1
Check: (+1) + 7(+7) + 4(-1) = 0
Formula: NaMnF4 
----------------------------------------


In [16]:
# Break the correct answers into those supported and not supported by the stated reasoning.
def counts(records):
    correct = [r for rec in records for r in rec["responses"] if score_response(r, rec["elements"]) == 1]
    supported = sum(stated_states_match(r) for r in correct)
    return {"Correct answers": len(correct), "Supported by reasoning": supported,
            "Not supported by reasoning": len(correct) - supported}

with open("data/baseline_responses.json") as f:
    b = counts(json.load(f))
with open("data/dpo_responses.json") as f:
    d = counts(json.load(f))
display(pd.DataFrame({"Baseline": b, "After DPO": d}))

,Baseline,After DPO
Correct answers,197,251
Supported by reasoning,62,57
Not supported by reasoning,135,194


In [18]:
##next step, evaluation analysis (where inside the answers training caused this result of better chemical formulation but a drop in reasoning support)
##using completely new questions to see if the drop in reasoning support persists
## table 1 shows how much more or less likely training made those answers as a whole
## table 2 shows the average-per-token delta (based on log probabilities)

In [7]:
# Load the original model with the trained adapter attached but NOT merged, so one copy can act
# as the trained model (adapter on) or the original model (adapter off).
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from peft import PeftModel

DEVICE = "mps"
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL_NAME, local_files_only=True)
tok.padding_side = "left"
base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float32, local_files_only=True)
model = PeftModel.from_pretrained(base, "adapters/dpo_beta0.1").to(DEVICE)
model.eval()

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 1536)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=1536, out_features=1536, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1536, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=1536, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear(
                (ba

In [9]:
# For one prompt and response, compute each response token's log-probability change from training
# (adapter on minus adapter off) and label which part of the answer the token belongs to.
import torch.nn.functional as F

def token_logprobs(ids, start):
    logits = model(ids).logits[0, start - 1:-1]
    return F.log_softmax(logits, dim=-1).gather(1, ids[0, start:, None]).squeeze(1)

def region_of(response, pos, metal_span):
    if metal_span[0] <= pos < metal_span[1]:
        return "Metal's stated state"
    line = response.split("\n")[response[:pos].count("\n")]
    if "Formula" in line:
        return "Formula line"
    if line.startswith("Check"):
        return "Check line"
    if re.search(r"\b[A-Z][a-z]?:\s*[+-]?\d", line):
        return "Other stated states"
    return "Other text"

@torch.no_grad()
def analyze(prompt, response, metal):
    prompt_text = build_input(prompt)
    enc = tok(prompt_text + response, return_offsets_mapping=True, return_tensors="pt",
              add_special_tokens=False)
    ids, offsets = enc["input_ids"].to(DEVICE), enc["offset_mapping"][0].tolist()
    start = next(i for i, (s, e) in enumerate(offsets) if s >= len(prompt_text))
    trained = token_logprobs(ids, start)
    with model.disable_adapter():
        original = token_logprobs(ids, start)
    m = re.search(rf"\b{metal}:\s*([+-]?\d+)", normalize(response))
    span = m.span(1) if m else (-1, -1)
    return [(region_of(response, offsets[start + k][0] - len(prompt_text), span), d)
            for k, d in enumerate((trained - original).tolist())]

In [ ]:
# Step-by-step timing of one analysis, printing after each stage to find where it stalls.
# THE TIMING ANALYSIS OF ONE PROMPT

import time
with open("data/test_prompts.json") as f:
    test = json.load(f)
with open("data/baseline_responses.json") as f:
    records = json.load(f)
t0 = time.time()
prompt_text = build_input(test[0]["prompt"])
enc = tok(prompt_text + records[0]["responses"][0], return_offsets_mapping=True,
          return_tensors="pt", add_special_tokens=False)
ids = enc["input_ids"].to(DEVICE)
print("1. tokenized:", ids.shape[1], "tokens at", round(time.time() - t0, 1), "s", flush=True)
with torch.no_grad():
    model(ids)
print("2. pass with adapter done at", round(time.time() - t0, 1), "s", flush=True)
with torch.no_grad(), model.disable_adapter():
    model(ids)
print("3. pass without adapter done at", round(time.time() - t0, 1), "s", flush=True)

1. tokenized: 214 tokens at 0.1 s
2. pass with adapter done at 0.3 s
3. pass without adapter done at 0.4 s


In [11]:
# Run the complete analysis function on one response, to test the region-labeling code too.
t0 = time.time()
out = analyze(test[0]["prompt"], records[0]["responses"][0], records[0]["elements"][1])
print(len(out), "tokens analyzed in", round(time.time() - t0, 1), "s")
for region, d in out[:12]:
    print(f"{region:22s} {d:+.3f}")

41 tokens analyzed in 23.1 s
Other stated states    +0.009
Other stated states    +0.000
Other stated states    +0.000
Other stated states    +0.000
Other stated states    +0.000
Other stated states    +0.000
Other stated states    +0.000
Other stated states    +0.000
Metal's stated state   -0.214
Other stated states    +0.001
Other stated states    +0.000
Other stated states    +0.000


In [12]:
# Time a second analysis (the first can include one-time GPU setup) and estimate the full run.
t0 = time.time()
analyze(test[1]["prompt"], records[1]["responses"][0], records[1]["elements"][1])
per = time.time() - t0
n = sum(len(set(rec["responses"])) for rec in records)
print(f"{per:.1f} s per response x {n} distinct responses = about {per * n / 60:.0f} minutes")

11.0 s per response x 179 distinct responses = about 33 minutes


In [13]:
# Actually run the analysis on every distinct held-out baseline response (written by the original model,
# never trained on), recording each token's change and its response's category.
import numpy as np
import pandas as pd

def category(r, elements):
    s = score_response(r, elements)
    if s == 1:
        return "Correct, supported" if stated_states_match(r) else "Correct, unsupported"
    return "Valid, not balanced" if s == 0 else "Unusable or wrong elements"

with open("data/test_prompts.json") as f:
    test = json.load(f)
with open("data/baseline_responses.json") as f:
    records = json.load(f)

rows = []
for i, (t, rec) in enumerate(zip(test, records)):
    for j, r in enumerate(dict.fromkeys(rec["responses"])):
        cat = category(r, rec["elements"])
        for region, d in analyze(t["prompt"], r, rec["elements"][1]):
            rows.append({"response": f"{i}-{j}", "category": cat, "region": region, "delta": d})
    print(f"{i + 1}/44 prompts done")
tokens = pd.DataFrame(rows)
tokens.to_csv("data/token_deltas.csv", index=False)

1/44 prompts done
2/44 prompts done
3/44 prompts done
4/44 prompts done
5/44 prompts done
6/44 prompts done
7/44 prompts done
8/44 prompts done
9/44 prompts done
10/44 prompts done
11/44 prompts done
12/44 prompts done
13/44 prompts done
14/44 prompts done
15/44 prompts done
16/44 prompts done
17/44 prompts done
18/44 prompts done
19/44 prompts done
20/44 prompts done
21/44 prompts done
22/44 prompts done
23/44 prompts done
24/44 prompts done
25/44 prompts done
26/44 prompts done
27/44 prompts done
28/44 prompts done
29/44 prompts done
30/44 prompts done
31/44 prompts done
32/44 prompts done
33/44 prompts done
34/44 prompts done
35/44 prompts done
36/44 prompts done
37/44 prompts done
38/44 prompts done
39/44 prompts done
40/44 prompts done
41/44 prompts done
42/44 prompts done
43/44 prompts done
44/44 prompts done


In [14]:
# Table 1: how much more or less likely training made each category of answer as a whole.
# Table 2: the average per-token change in each part of the answer, within each category.
per_response = tokens.groupby(["category", "response"])["delta"].sum().groupby("category")
summary = pd.DataFrame({"Distinct responses": per_response.size(),
                        "Avg total log-prob change": per_response.mean().round(2)})
summary["Likelihood ratio (trained ÷ original)"] = np.exp(summary["Avg total log-prob change"]).map("{:.3g}×".format)
display(summary)

regions = ["Metal's stated state", "Other stated states", "Check line", "Formula line", "Other text"]
by_region = tokens.pivot_table(index="category", columns="region", values="delta", aggfunc="mean")
display(by_region.reindex(columns=regions).round(3))

,Distinct responses,Avg total log-prob change,Likelihood ratio (trained ÷ original)
category,,,
"Correct, supported",24,-0.50,0.607×
"Correct, unsupported",74,-0.11,0.896×
Unusable or wrong elements,30,-3.94,0.0194×
"Valid, not balanced",51,-1.12,0.326×


region,Metal's stated state,Other stated states,Check line,Formula line,Other text
category,,,,,
"Correct, supported",-0.056,0.000,-0.036,0.046,NaN
"Correct, unsupported",-0.167,0.002,-0.032,0.101,0.055
Unusable or wrong elements,-0.119,-0.000,-0.082,-0.325,NaN
"Valid, not balanced",-0.151,0.001,-0.042,-0.016,NaN


In [15]:
# Break each category's total change into contributions from each region of the answer.
# The Total column should match Table 1's "Avg total log-prob change".
per = tokens.groupby(["category", "response", "region"])["delta"].sum().unstack(fill_value=0)
contrib = per.groupby("category").mean().reindex(columns=regions).fillna(0)
contrib["Total"] = contrib.sum(axis=1)
display(contrib.round(2))

region,Metal's stated state,Other stated states,Check line,Formula line,Other text,Total
category,,,,,,
"Correct, supported",-0.06,0.00,-0.75,0.31,0.00,-0.50
"Correct, unsupported",-0.17,0.03,-0.67,0.67,0.03,-0.11
Unusable or wrong elements,-0.12,-0.00,-1.70,-2.12,0.00,-3.94
"Valid, not balanced",-0.15,0.02,-0.88,-0.11,0.00,-1.12


In [5]:
# Generate the trained model's answers for the 176 training combinations, so every combination on
# the website has answers from both models. Same settings as before; saves after every 16 prompts.
with open("data/train_prompts.json") as f:
    train = json.load(f)
path = "data/dpo_train_responses.json"
done = json.load(open(path)) if os.path.exists(path) else []

set_seed(0)
for start in range(len(done), len(train), 16):
    chunk = train[start:start + 16]
    groups = generate([t["prompt"] for t in chunk])
    done += [{"elements": t["elements"], "responses": g} for t, g in zip(chunk, groups)]
    with open(path, "w") as f:
        json.dump(done, f, indent=2)
    print(f"Saved {len(done)} of {len(train)} prompts")

4/16 prompts done
8/16 prompts done
12/16 prompts done
16/16 prompts done
Saved 16 of 176 prompts
4/16 prompts done
8/16 prompts done
12/16 prompts done
16/16 prompts done
Saved 32 of 176 prompts
4/16 prompts done
8/16 prompts done
12/16 prompts done
16/16 prompts done
Saved 48 of 176 prompts
4/16 prompts done
8/16 prompts done
12/16 prompts done
16/16 prompts done
Saved 64 of 176 prompts
4/16 prompts done
8/16 prompts done
12/16 prompts done
16/16 prompts done
Saved 80 of 176 prompts
4/16 prompts done
8/16 prompts done
12/16 prompts done
16/16 prompts done
Saved 96 of 176 prompts
4/16 prompts done
8/16 prompts done
12/16 prompts done
16/16 prompts done
Saved 112 of 176 prompts
4/16 prompts done
8/16 prompts done
12/16 prompts done
16/16 prompts done
Saved 128 of 176 prompts
4/16 prompts done
8/16 prompts done
12/16 prompts done
16/16 prompts done
Saved 144 of 176 prompts
4/16 prompts done
8/16 prompts done
12/16 prompts done
16/16 prompts done
Saved 160 of 176 prompts
4/16 prompts don

In [6]:
# Combine both models' answers for all 220 combinations into one file for the website's API.
def load_answers(path):
    with open(path) as f:
        return {"-".join(r["elements"]): r["responses"] for r in json.load(f)}

original = {**load_answers("data/train_responses.json"), **load_answers("data/baseline_responses.json")}
trained = {**load_answers("data/dpo_train_responses.json"), **load_answers("data/dpo_responses.json")}
with open("data/test_prompts.json") as f:
    held_out = {"-".join(t["elements"]) for t in json.load(f)}

answers = {k: {"split": "held-out" if k in held_out else "training",
               "original": original[k], "dpo": trained[k]} for k in original}
with open("data/answers.json", "w") as f:
    json.dump(answers, f)
print(len(answers), "combinations,", sum(v["split"] == "held-out" for v in answers.values()), "held-out")

220 combinations, 44 held-out
